In [4]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR

# ===============================
# 설정
# ===============================

DATA_PATH = "./merged_var_input.csv"

VARIABLES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_DXY",
    "d_UST10Y",
    "dlog_VIX"
]

LAG_ORDER = 1
FEVD_HORIZON = 20

OUTPUT_FEVD = "./fevd_results.csv"
OUTPUT_FEVD_FINAL = "./fevd_final_horizon.csv"


# ===============================
# 데이터 로드
# ===============================

df = pd.read_csv(DATA_PATH)

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

data = df[VARIABLES].dropna()

print("Data shape:", data.shape)


# ===============================
# VAR 적합
# ===============================

model = VAR(data)
results = model.fit(LAG_ORDER)

print("\nVAR fitted successfully")


# ===============================
# FEVD 계산
# ===============================

fevd = results.fevd(FEVD_HORIZON)

print("\nfevd.decomp.shape =", fevd.decomp.shape)

n_vars = len(VARIABLES)

# statsmodels 버전에 따라 실제 horizon 길이가 요청값과 다를 수 있으므로 안전하게 처리
actual_horizon = fevd.decomp.shape[1]
print("Actual FEVD horizon =", actual_horizon)


# ===============================
# horizon별 long-format 저장
# ===============================

rows = []

for h in range(actual_horizon):
    # shape: (response, shock)
    matrix = fevd.decomp[:, h, :]

    for i, response in enumerate(VARIABLES):
        for j, shock in enumerate(VARIABLES):
            rows.append({
                "horizon": h + 1,
                "response": response,
                "shock": shock,
                "fevd": matrix[i, j]
            })

fevd_df = pd.DataFrame(rows)
fevd_df.to_csv(OUTPUT_FEVD, index=False)

print("\nSaved:", OUTPUT_FEVD)


# ===============================
# 최종 horizon FEVD matrix 저장
# ===============================

final_matrix = fevd.decomp[:, actual_horizon - 1, :]

fevd_final = pd.DataFrame(
    final_matrix,
    index=VARIABLES,
    columns=VARIABLES
)

fevd_final.to_csv(OUTPUT_FEVD_FINAL)

print("\nSaved:", OUTPUT_FEVD_FINAL)
print("\nFinal horizon FEVD matrix:")
print(fevd_final.round(6))


# ===============================
# 관심 관계 출력
# ===============================

solvpn_from_copper = fevd_final.loc["dlog_SOLVPN", "dlog_COPPER"]
copper_from_solvpn = fevd_final.loc["dlog_COPPER", "dlog_SOLVPN"]

print("\n===== Key FEVD entries =====")
print(f"COPPER -> SOLVPN : {solvpn_from_copper:.6f}")
print(f"SOLVPN -> COPPER : {copper_from_solvpn:.6f}")


# ===============================
# 각 변수별 외부 충격 합
# ===============================

print("\n===== FROM others =====")
for var in VARIABLES:
    from_others = fevd_final.loc[var].sum() - fevd_final.loc[var, var]
    print(f"{var}: {from_others:.6f}")

Data shape: (1325, 5)

VAR fitted successfully

fevd.decomp.shape = (5, 20, 5)
Actual FEVD horizon = 20

Saved: ./fevd_results.csv

Saved: ./fevd_final_horizon.csv

Final horizon FEVD matrix:
             dlog_SOLVPN  dlog_COPPER  dlog_DXY  d_UST10Y  dlog_VIX
dlog_SOLVPN     0.994333     0.000041  0.000178  0.002775  0.002673
dlog_COPPER     0.053676     0.937693  0.001371  0.000056  0.007204
dlog_DXY        0.097844     0.070306  0.821295  0.006938  0.003617
d_UST10Y        0.052877     0.003068  0.086755  0.856811  0.000490
dlog_VIX        0.283188     0.015240  0.002987  0.011607  0.686978

===== Key FEVD entries =====
COPPER -> SOLVPN : 0.000041
SOLVPN -> COPPER : 0.053676

===== FROM others =====
dlog_SOLVPN: 0.005667
dlog_COPPER: 0.062307
dlog_DXY: 0.178705
d_UST10Y: 0.143189
dlog_VIX: 0.313022
